# Read Images, Generate Medsam Image, and Write to Disk

In [ ]:
# !pip install -r requirements.txt --use-deprecated=legacy-resolver
!pip install ipykernel==5.5.6 google-colab
!pip install -q git+https://github.com/bowang-lab/MedSAM.git


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 47.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.4/52.4 MB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.3/516.3 kB 29.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 72.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 57.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.2/117.2 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.6/383.6 kB 27.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.7/59.7 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.5/133.5 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.4/66.4 kB 6.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages th

In [ ]:
!pip install segment_anything

In [ ]:
# %% environment and functions
import numpy as np
import matplotlib.pyplot as plt
import os
join = os.path.join
import torch
from segment_anything import sam_model_registry
from skimage import io, transform
import torch.nn.functional as F

In [ ]:
# visualization functions
# source: https://github.com/facebookresearch/segment-anything/blob/main/notebooks/predictor_example.ipynb
# change color to avoid red and green
def show_mask(mask, ax, random_color=False):
    if random_color:
        color = np.concatenate([np.random.random(3), np.array([0.6])], axis=0)
    else:
        color = np.array([251/255, 252/255, 30/255, 0.6])
    h, w = mask.shape[-2:]
    mask_image = mask.reshape(h, w, 1) * color.reshape(1, 1, -1)
    ax.imshow(mask_image)

def show_box(box, ax):
    x0, y0 = box[0], box[1]
    w, h = box[2] - box[0], box[3] - box[1]
    ax.add_patch(plt.Rectangle((x0, y0), w, h, edgecolor='blue', facecolor=(0,0,0,0), lw=2))

@torch.no_grad()
def medsam_inference(medsam_model, img_embed, box_1024, H, W):
    box_torch = torch.as_tensor(box_1024, dtype=torch.float, device=img_embed.device)
    if len(box_torch.shape) == 2:
        box_torch = box_torch[:, None, :] # (B, 1, 4)

    sparse_embeddings, dense_embeddings = medsam_model.prompt_encoder(
        points=None,
        boxes=box_torch,
        masks=None,
    )
    low_res_logits, _ = medsam_model.mask_decoder(
        image_embeddings=img_embed, # (B, 256, 64, 64)
        image_pe=medsam_model.prompt_encoder.get_dense_pe(), # (1, 256, 64, 64)
        sparse_prompt_embeddings=sparse_embeddings, # (B, 2, 256)
        dense_prompt_embeddings=dense_embeddings, # (B, 256, 64, 64)
        multimask_output=False,
        )

    low_res_pred = torch.sigmoid(low_res_logits)  # (1, 1, 256, 256)

    low_res_pred = F.interpolate(
        low_res_pred,
        size=(H, W),
        mode="bilinear",
        align_corners=False,
    )  # (1, 1, gt.shape)
    low_res_pred = low_res_pred.squeeze().cpu().numpy()  # (256, 256)
    medsam_seg = (low_res_pred > 0.5).astype(np.uint8)
    return medsam_seg


In [ ]:
# download model and data
!wget -O img_demo.png https://raw.githubusercontent.com/bowang-lab/MedSAM/main/assets/img_demo.png
!wget -O medsam_vit_b.pth https://zenodo.org/records/10689643/files/medsam_vit_b.pth

--2024-11-14 04:58:00--  https://raw.githubusercontent.com/bowang-lab/MedSAM/main/assets/img_demo.png
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 87865 (86K) [image/png]
Saving to: ‘img_demo.png’

img_demo.png        100%[===================>]  85.81K  --.-KB/s    in 0.003s  

2024-11-14 04:58:00 (30.5 MB/s) - ‘img_demo.png’ saved [87865/87865]

--2024-11-14 04:58:00--  https://zenodo.org/records/10689643/files/medsam_vit_b.pth
Resolving zenodo.org (zenodo.org)... 188.185.79.172, 188.184.103.159, 188.184.98.238, ...
Connecting to zenodo.org (zenodo.org)|188.185.79.172|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 375049145 (358M) [application/octet-stream]
Saving to: ‘medsam_vit_b.pth’

medsam_vit_b.pth    100%[===============

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import matplotlib.patches as patch
import cv2
from google.colab.patches import cv2_imshow
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import pickle
import os

In [ ]:
file_path='/content/drive/MyDrive/bowen_research/mridata/metadata.csv'
output_path='/content/drive/MyDrive/bowen_research/mridata/segmented/'
input_folder_path='/content/drive/MyDrive/bowen_research/mridata/'
combined_input_path='/content/drive/MyDrive/bowen_research/mridata/combined/'
volumetric_data_dir='/content/drive/MyDrive/bowen_research/mridata/vol01/'
metadata = np.genfromtxt(file_path, delimiter=',', names=True,
    dtype='i4,i4,i4,i4,i4,i4,i4,i4,i4,i4,U20')

In [ ]:
# List of subfolders containing the files you want to copy
subfolders = ["/content/drive/MyDrive/bowen_research/mridata/vol0"+str(x) for x in range(2,9) ]

In [ ]:
subfolders

['/content/drive/MyDrive/bowen_research/mridata/vol02',
 '/content/drive/MyDrive/bowen_research/mridata/vol03',
 '/content/drive/MyDrive/bowen_research/mridata/vol04',
 '/content/drive/MyDrive/bowen_research/mridata/vol05',
 '/content/drive/MyDrive/bowen_research/mridata/vol06',
 '/content/drive/MyDrive/bowen_research/mridata/vol07',
 '/content/drive/MyDrive/bowen_research/mridata/vol08']

In [ ]:
import os
import shutil

# Destination folder where all files will be copied
destination_folder = combined_input_path

# Ensure the destination folder exists
os.makedirs(destination_folder, exist_ok=True)

# Copy each file from each subfolder to the destination folder
for subfolder in subfolders:
    for filename in os.listdir(subfolder):
        file_path = os.path.join(subfolder, filename)
        if os.path.isfile(file_path):  # Check if it's a file
            shutil.copy(file_path, destination_folder)

print("All files have been copied successfully.")

All files have been copied successfully.


In [ ]:
len(metadata)

917

In [ ]:
metadata[0]

(329637, 8, 0, 1, 139, 184, 14, 74, 72, 3, '329637-8.pck')

In [ ]:
metadata[99]

(504699, 5, 0, 0, 130, 122, 11, 91, 95, 4, '504699-5.pck')

In [ ]:
combined_input_path

'/content/drive/MyDrive/bowen_research/mridata/combined/'

In [ ]:
img_paths = np.empty(0)
rois = np.empty((0, 4))
for exam in metadata[200:]:
  vol_data_file = exam['volumeFilename']
  vol_data_path = os.path.join(combined_input_path, vol_data_file)
  if os.path.exists(vol_data_path):
    with open(vol_data_path, 'rb') as f:
      loaded_image_array=pickle.load(f)
    z_start = exam['roiZ']
    depth = exam['roiDepth']
    saveString = ''
    roi = [exam['roiX'], exam['roiY'], exam['roiWidth'], exam['roiHeight']]
    for z in range(z_start, z_start + depth):
      slice = loaded_image_array[z, :, :]
      saveString = vol_data_file[:6] + '-' + str(z-z_start) + '.jpg'
      img_paths = np.append(img_paths, saveString)
      rois = np.vstack([rois, roi])
      output_path = '/content/drive/MyDrive/bowen_research/mridata/jpgs/'
      # change page to output_path
      output_file = os.path.join(output_path, saveString)
      print(output_file)
      plt.imsave(output_file, slice, format='jpg', cmap = 'gray')

/content/drive/MyDrive/bowen_research/mridata/jpgs/566015-0.jpg
/content/drive/MyDrive/bowen_research/mridata/jpgs/566015-1.jpg
/content/drive/MyDrive/bowen_research/mridata/jpgs/566015-2.jpg
/content/drive/MyDrive/bowen_research/mridata/jpgs/566022-0.jpg
/content/drive/MyDrive/bowen_research/mridata/jpgs/566022-1.jpg
/content/drive/MyDrive/bowen_research/mridata/jpgs/566022-2.jpg
/content/drive/MyDrive/bowen_research/mridata/jpgs/566069-0.jpg
/content/drive/MyDrive/bowen_research/mridata/jpgs/566069-1.jpg
/content/drive/MyDrive/bowen_research/mridata/jpgs/566069-2.jpg
/content/drive/MyDrive/bowen_research/mridata/jpgs/566069-3.jpg
/content/drive/MyDrive/bowen_research/mridata/jpgs/566082-0.jpg
/content/drive/MyDrive/bowen_research/mridata/jpgs/566082-1.jpg
/content/drive/MyDrive/bowen_research/mridata/jpgs/566082-2.jpg
/content/drive/MyDrive/bowen_research/mridata/jpgs/566089-0.jpg
/content/drive/MyDrive/bowen_research/mridata/jpgs/566089-1.jpg
/content/drive/MyDrive/bowen_research/mr

In [ ]:
import os
import glob

# Path to the folder where JPG files are stored
folder_path = "/content/"

# Find all .jpg files in the folder
jpg_files = glob.glob(os.path.join(folder_path, "*.jpg"))

# Delete each file
for file_path in jpg_files:
    print(f"Deleting file: {file_path}")
    os.remove(file_path)

# print("All JPG files have been deleted.")

Deleting file: /content/538749-0.jpg
Deleting file: /content/860375-1.jpg
Deleting file: /content/602331-0.jpg
Deleting file: /content/905220-2.jpg
Deleting file: /content/799331-1.jpg
Deleting file: /content/852918-0.jpg
Deleting file: /content/549879-2.jpg
Deleting file: /content/547422-0.jpg
Deleting file: /content/919200-1.jpg
Deleting file: /content/515999-3.jpg
Deleting file: /content/638165-2.jpg
Deleting file: /content/670898-3.jpg
Deleting file: /content/588775-0.jpg
Deleting file: /content/579840-0.jpg
Deleting file: /content/919778-1.jpg
Deleting file: /content/740062-1.jpg
Deleting file: /content/742863-0.jpg
Deleting file: /content/566015-1.jpg
Deleting file: /content/783758-2.jpg
Deleting file: /content/618174-0.jpg
Deleting file: /content/879816-1.jpg
Deleting file: /content/689147-1.jpg
Deleting file: /content/563519-1.jpg
Deleting file: /content/560649-1.jpg
Deleting file: /content/500804-2.jpg
Deleting file: /content/742110-2.jpg
Deleting file: /content/754074-0.jpg
D

In [ ]:
#%% load model and image
MedSAM_CKPT_PATH = "/content/medsam_vit_b.pth"
device = "cuda:0"
medsam_model = sam_model_registry['vit_b'](checkpoint=MedSAM_CKPT_PATH)
medsam_model = medsam_model.to(device)
medsam_model.eval()

/usr/local/lib/python3.10/dist-packages/segment_anything/build_sam.py:105: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(f)


Sam(
  (image_encoder): ImageEncoderViT(
    (patch_embed): PatchEmbed(
      (proj): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
    )
    (blocks): ModuleList(
      (0-11): 12 x Block(
        (norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
        (attn): Attention(
          (qkv): Linear(in_features=768, out_features=2304, bias=True)
          (proj): Linear(in_features=768, out_features=768, bias=True)
        )
        (norm2): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
        (mlp): MLPBlock(
          (lin1): Linear(in_features=768, out_features=3072, bias=True)
          (lin2): Linear(in_features=3072, out_features=768, bias=True)
          (act): GELU(approximate='none')
        )
      )
    )
    (neck): Sequential(
      (0): Conv2d(768, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (1): LayerNorm2d()
      (2): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (3): LayerNorm2d()
    )


In [ ]:
def enlarge_box(box, scale_factor=1):
    x1, y1, x2, y2 = box
    width = x2 - x1
    height = y2 - y1
    center_x = x1 + width / 2
    center_y = y1 + height / 2

    new_width = width * scale_factor
    new_height = height * scale_factor

    new_x1 = max(center_x - new_width / 2, 0)
    new_y1 = max(center_y - new_height / 2, 0)
    new_x2 = min(center_x + new_width / 2, 1024)
    new_y2 = min(center_y + new_height / 2, 1024)

    return np.array([[new_x1, new_y1, new_x2, new_y2]])

In [ ]:
def find_corners(roiX, roiY, roiWidth, roiHeight):
  return int(roiX), int(roiY), int(roiX + roiWidth), int(roiY + roiHeight)

In [ ]:
img_paths.shape

(1838,)

In [ ]:
img_paths[0]

'566015-0.jpg'

In [ ]:
knees=list(set(img.split('-')[0] for img in img_paths))

In [ ]:
len(knees)

530

In [ ]:
import random

In [ ]:
random.choice([1,2,3])

3

In [ ]:
output_path

'/content/drive/MyDrive/bowen_research/mridata/jpgs/'

In [ ]:
jpg_path=output_path

In [ ]:
segmented_path='/content/drive/MyDrive/bowen_research/mridata/segmented/'

In [ ]:
len(rois)

1838

In [ ]:
img_paths

array(['566015-0.jpg', '566015-1.jpg', '566015-2.jpg', ...,
       '919795-1.jpg', '919795-2.jpg', '919795-3.jpg'], dtype='<U32')

In [ ]:
for k in knees:
  print(k)
  # randomly choose one image from images in img_paths that match i from the beginning
  imgs=[img for img in img_paths if img.startswith(k)]
  rand_img=random.choice(imgs)
  print(rand_img)
  # find the index of the random image in img_paths
  i=img_paths.tolist().index(rand_img)
  print(i)

  image = cv2.imread(jpg_path+rand_img)
  img_np=image
  if len(img_np.shape) == 2:
        img_3c = np.repeat(img_np[:, :, None], 3, axis=-1)
        H, W, _ = img_3c.shape
  else:
      img_3c = img_np
      H, W, _ = img_3c.shape
  #%% image preprocessing and model inference
  img_1024 = transform.resize(img_3c, (1024, 1024), order=3, preserve_range=True, anti_aliasing=True).astype(np.uint8)
  img_1024 = (img_1024 - img_1024.min()) / np.clip(
      img_1024.max() - img_1024.min(), a_min=1e-8, a_max=None
  )  # normalize to [0, 1], (H, W, 3)
  # convert the shape to (3, H, W)
  img_1024_tensor = torch.tensor(img_1024).float().permute(2, 0, 1).unsqueeze(0).to(device)

  box_np = find_corners(rois[i][0],rois[i][1],rois[i][2],rois[i][3])


  box_1024 = box_np / np.array([W, H, W, H]) * 1024

  box_1024_enlarged = enlarge_box(box_1024, scale_factor=1)
    # box_1024_with_margin = add_margin_to_box(box_1024, margin=20)

    # Inference using the enlarged bounding box
  with torch.no_grad():
      image_embedding = medsam_model.image_encoder(img_1024_tensor)  # (1, 256, 64, 64)

  medsam_seg = medsam_inference(medsam_model, image_embedding, box_1024_enlarged, H, W)
  fig, ax = plt.subplots(1, 2, figsize=(10, 5))
  ax[0].imshow(img_3c)
  ax[0].set_title("Input Image")
  ax[1].imshow(img_3c)
  ax[1].set_title("MedSAM Segmentation")
  show_mask(medsam_seg, ax[1])
  show_box(box_np, ax[1])
  # Extract the file name (without extension) from the original image path
  original_file_name = os.path.splitext(os.path.basename(img_paths[i]))[0]

  # Create the output path using the original file name
  output_path_file = f"{segmented_path}/{original_file_name}_segmented.jpg"

  # Save the figure containing both images
  plt.savefig(output_path_file,format='jpg')

  # Optional: close the figure to free up memory
  plt.close(fig)

800663
800663-1.jpg
1178
863766
863766-0.jpg
1500
769102
769102-0.jpg
970
847907
847907-2.jpg
1385
622688
622688-2.jpg
360
895625
895625-1.jpg
1674
566089
566089-0.jpg
13
733198
733198-0.jpg
756
852918
852918-2.jpg
1397
915722
915722-1.jpg
1771
624797
624797-0.jpg
370
688886
688886-0.jpg
634
593613
593613-2.jpg
197
601122
601122-2.jpg
260
791130
791130-0.jpg
1123
872088
872088-2.jpg
1536
785256
785256-3.jpg
1084
766889
766889-1.jpg
950
858210
858210-2.jpg
1429
774080
774080-1.jpg
997
905147
905147-2.jpg
1722
642940
642940-2.jpg
470
685842
685842-0.jpg
618
858166
858166-1.jpg
1425
780871
780871-2.jpg
1052
888167
888167-2.jpg
1613
616731
616731-2.jpg
320
652255
652255-0.jpg
505
690314
690314-0.jpg
640
903145
903145-1.jpg
1713
895196
895196-2.jpg
1665
569264
569264-2.jpg
45
580017
580017-2.jpg
90
749555
749555-2.jpg
837
582730
582730-1.jpg
118
886869
886869-2.jpg
1598
598322
598322-2.jpg
233
590642
590642-0.jpg
178
588912
588912-4.jpg
174
717996
717996-1.jpg
704
717337
717337-1.jpg
698
86

In [ ]:
# read the file names from segmented_path
import os
file_names = os.listdir(segmented_path)
# use the first part of the file name to find the corresponding aclDiagnosis in metadata, then add it to the file name before .jpg
for file_name in file_names[:2]:
  # find the corresponding aclDiagnosis in metadata
  knee_number=file_name.split('-')[0]

  diag=metadata[metadata['volumeFilename'].split('-')[0]==knee_number]['aclDiagnosis']
  # add diag to the file name before .jpg
  new_name=filename.split('.')[0]+'_'+diag+'_.'+filename.split('.')[1]
  # rename the file
  print(os.path.join(segmented_path, file_name))
  print(os.path.join(segmented_path, new_name))
  # os.rename(os.path.join(segmented_path, file_name), os.path.join(segmented_path, new_name))

AttributeError: 'numpy.ndarray' object has no attribute 'split'

In [ ]:
metadata[metadata['volumeFilename']=='329637-8.pck']

array([(329637, 8, 0, 1, 139, 184, 14, 74, 72, 3, '329637-8.pck')],
      dtype=[('examId', '<i4'), ('seriesNo', '<i4'), ('aclDiagnosis', '<i4'), ('kneeLR', '<i4'), ('roiX', '<i4'), ('roiY', '<i4'), ('roiZ', '<i4'), ('roiHeight', '<i4'), ('roiWidth', '<i4'), ('roiDepth', '<i4'), ('volumeFilename', '<U20')])